# Benchmarking + Metrics Baseline
## Problem Statement
Run a 20-query structured benchmark over DevContext AI and record the 7 production metrics that will appear in the README.

## My Approach
 
Before touching any eval code I decided to separate the two expensive phases: agent runs first, DeepEval scoring second. I'd already seen from previous sprints how quickly Groq's TPM resets, so saving `benchmark_final_results.json` right after all queries finished was a deliberate checkpoint - if DeepEval hit rate limits, I wouldn't have to re-run the whole agent. 

I also added a raw Groq SDK call purely to read the rate-limit headers before starting, so I knew upfront whether I had headroom. The three separate JSON artifacts (`benchmark_final_results.json`, `faithfulness_final_results.json`, `metrics_baseline.json`) were a conscious choice: each stage of the pipeline saves its own evidence, so I can inspect what went wrong at exactly the right layer without re-running everything.
 
## What I Learned
 
The most surprising thing was how much the rate-limit problem forced better architecture decisions. Storing intermediate results as versioned JSON files isn't just a workaround - it's how production evaluation pipelines actually work. 

You don't run 20 agent queries and 20 DeepEval calls in one shot and hope nothing fails. You checkpoint. 

`time.perf_counter()` being elapsed seconds since an arbitrary point (not a wall-clock timestamp) was a genuine gotcha — it looked right in output until you tried to read it as a date.

## Where I Got Stuck
 
Rate limits hit harder than expected. The Groq free tier throttles both on tokens-per-minute and daily requests, and running the full agent pipeline (which makes multiple LLM calls per query — rewrite, intent route, parallel router, generate) means each "query" burns 3–5 LLM calls, not one. 

I had to slow everything down with `time.sleep(5)` between queries, and I still hit the daily cap before completing the full 20-query set. This is why I stored intermediate results: to avoid losing everything when the daily limit reset. The rate-limit mitigation forced me to restructure the notebook into discrete save-checkpoints, which ended up being cleaner architecture anyway.
 
## What I'd Do Differently
 
I'd instrument the ingestion blocklist at the point it fires - log blocked file paths to a separate JSON during ingest, then just read that file in the benchmark notebook rather than hardcoding the count. 

I would add chunk boundary detection: the approach (checking if the answer ends with a non-sentence-ending character) would too simple - a better approach would be to flag cases where `retrieved_context` contains a hard cut-off mid-function-body, which is detectable at the retrieval stage rather than at the answer stage.
 
On the rate-limit problem: if I were doing this again from the start, I'd add a token-budget tracker to `run_devcontext_agent` itself — counting approximate tokens consumed per run and backing off automatically when close to the TPM limit. That would make the benchmark runner self-throttling instead of relying on hardcoded `sleep` calls.

In [1]:
import time
from devcontext_agent import run_devcontext_agent


d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
# DIAGNOSTIC ONLY — raw Groq SDK used here solely to inspect rate-limit headers.
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()  
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.with_raw_response.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{"role": "user", "content": "Hello"}]
)

# Extract Groq's tracking headers
headers = response.headers
print(f"Remaining Tokens this minute: {headers.get('x-ratelimit-remaining-tokens')}")
print(f"Time until TPM resets: {headers.get('x-ratelimit-reset-tokens')}")
print(f"Time until Daily Requests reset: {headers.get('x-ratelimit-reset-requests')}")

Remaining Tokens this minute: 29988
Time until TPM resets: 24ms
Time until Daily Requests reset: 1m26.4s


In [3]:
#Test run_devcontext_agent
test_queries = [
    "How to configure CORS in FastAPI?"
]

for query in test_queries:
    state, metrics = run_devcontext_agent(query , 'meta-llama/llama-4-scout-17b-16e-instruct')
    print(f"Query: {query}")
    print("-" * 40)
    print(f"Sources cited: {state.get("sources_cited","")}")
    print(f"Sources : {state.get("sources","")}")
    print(f"Retrieved Context : {state.get("retrieved_context","")}")
    print("-" * 40)
    print(f"Answer : {state.get("answer","")}")
    print(metrics)
    time.sleep(2)

Query: How to configure CORS in FastAPI?
----------------------------------------
Sources cited: ['codebase:setup', 'docs:auth-flow.md']
Sources : {'codebase': [{'id': 'code_fastapi_exceptions.py_chk0', 'text': 'def __init__(\n        self,\n        status_code: Annotated[\n            int,\n            Doc(\n                """\n                HTTP status code to send to the client.\n\n                Read more about it in the\n                [FastAPI docs for Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception)\n                """\n            ),\n        ],\n        detail: Annotated[\n            Any,\n            Doc(\n                """\n                Any data to be sent to the client in the `detail` key of the JSON\n                response.\n\n                Read more about it in the\n                [FastAPI docs for Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception)\n                """\n    

In [5]:

from langchain_groq import ChatGroq
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)

import time
import json
import statistics
from deepeval.metrics import FaithfulnessMetric , AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM


QA_QUERIES = [
    "How does FastAPI handle dependency injection?",
    "What does the Depends() function do?",
    "How does FastAPI validate request bodies?",
    "What is the role of APIRouter?",
    "How does FastAPI handle authentication?",
    "What does get_db return and where is it used?",
    "How does FastAPI generate OpenAPI docs automatically?",
    "What happens if a dependency raises an exception?",
    "How does BackgroundTasks work in FastAPI?",
    "What is the difference between path params and query params in FastAPI?",
]

PR_DIFF_SAMPLE = """
- def get_current_user(token: str = Depends(oauth2_scheme)):
+ def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
"""

In [6]:
#Benchmark runner
results=[]
for (i ,query) in enumerate(QA_QUERIES):
    start = time.perf_counter()
    state, metrics = run_devcontext_agent(query , 'meta-llama/llama-4-scout-17b-16e-instruct')

    print(f"Query [{i+1}]: {query}")
    elapsed_ms = (time.perf_counter() - start) * 1000
    results.append({
        "query": query
        ,"latency_ms": elapsed_ms
        ,"sources": state.get('sources','')
        ,"sources_cited": state.get('sources_cited','')
        ,"answer": state.get('answer','')
        ,"retrieved_context" : state.get('retrieved_context','')
        ,"hit_graceful_degradation": True if state.get("retry_attempt", 0) > 0 else False
    })
    print(f"Answer : {state.get('answer','No response text generated.')}")
    print("-" * 40)
    time.sleep(5)

Query [1]: How does FastAPI handle dependency injection?
Answer : FastAPI's dependency injection mechanism is implemented through the `Depends` and `Security` functions. The `Depends` function is used to declare a dependency, which is a "dependable" callable that FastAPI will call automatically. The `Security` function is similar to `Depends`, but it also allows for the declaration of OAuth2 scopes. The `analyze_param` function is used to extract information from the dependency, such as the dependency function, use cache, and scope. The dependency injection mechanism uses a concept called "dependable" callables, which are functions that can be called by FastAPI to resolve dependencies. The mechanism also supports features like caching, sub-dependencies, and scopes.
----------------------------------------
Query [2]: What does the Depends() function do?
Answer : The `Depends` function in the codebase is used to declare a FastAPI dependency, which is a "dependable" callable (like a funct

In [7]:
import json
output_filename = "benchmark_final_results.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f" Saved {len(results)} results permanently to {output_filename}!")

 Saved 10 results permanently to benchmark_final_results.json!


In [22]:
class GroqEvalJudge(DeepEvalBaseLLM):
    def __init__(self, model_name="meta-llama/llama-4-scout-17b-16e-instruct"):
        self.model_name = model_name

    def load_model(self):
        # This is where the actual API hit happens. It safely calls Llama 4 Scout.
        return ChatGroq(model=self.model_name, temperature=0)

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = chat_model.invoke(prompt)
        return res.content

    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content

    def get_model_name(self) -> str:
        # TRICK DEEPEVAL: Return a generic standard name so DeepEval's 
        # schema engines format the rubric correctly without throwing errors.
        return "llama3"
    
groq_judge = GroqEvalJudge()

faithfulness = FaithfulnessMetric(threshold=0.75, model=groq_judge, include_reason=True)
relevancy    = AnswerRelevancyMetric(threshold=0.75, model=groq_judge, include_reason=True)


In [23]:
#Compute metrics
latencies = [r["latency_ms"] for r in results]
p50 = statistics.median(latencies)
p95 = sorted(latencies)[int(len(latencies) * 0.95)]

In [25]:
#Faithfulness
faithful_score=[]
for case in [results[0]] :
    # print(type(case.get('retrieved_context' , '')))

    retrieval_list = [chunk.strip() for chunk in case.get('retrieved_context' , '').split("\n\n") if chunk.strip()]
    # print(len(retrieval_list))
    test_case = LLMTestCase(
    input=case.get('query' , ''),
    actual_output=case.get('answer' , ''),
    retrieval_context=retrieval_list
    )
    
    faithfulness.measure(test_case)
    relevancy.measure(test_case)

    print("=== Faithfulness ===")
    print(f"Score:  {faithfulness.score:.2f}")
    print(f"Passed: {faithfulness.is_successful()}")
    print(f"Reason: {faithfulness.reason}\n")

    print("=== Answer Relevancy ===")
    print(f"Score:  {relevancy.score:.2f}")
    print(f"Passed: {relevancy.is_successful()}")
    print(f"Reason: {relevancy.reason}")

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

=== Faithfulness ===
Score:  0.83
Passed: True
Reason: The score is 0.83 because despite the retrieval context providing relevant information, the actual output incorrectly references an 'analyze_param' function not mentioned in the context, indicating a minor deviation.

=== Answer Relevancy ===
Score:  0.88
Passed: True
Reason: The score is 0.88 because while the output largely addresses FastAPI's dependency injection, a specific mention of 'analyze_param' seems out of place and unrelated to the topic, slightly detracting from the overall relevancy.


In [ ]:
#Faithfulness
faithful_score=[]
for case in results :
    retrieval_list = [chunk.strip() for chunk in case.get('retrieved_context' , '').split("\n\n") if chunk.strip()]
    
    test_case = LLMTestCase(
    input=case.get('query' , ''),
    actual_output=case.get('answer' , ''),
    retrieval_context=retrieval_list
    )
    
    faithfulness.measure(test_case)
    
    faithful_score.append(
        {
        'result': case  
        ,'Score':  faithfulness.score
        ,'Passed': faithfulness.is_successful()
        ,'Reason': faithfulness.reason
        }
    )
    time.sleep(5)
    

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [27]:
import json
output_filename = "faithfulness_final_results.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(faithful_score, f, indent=4, ensure_ascii=False)

print(f" Saved {len(results)} results permanently to {output_filename}!")

 Saved 10 results permanently to faithfulness_final_results.json!


In [29]:
score_faithful = [f["Score"] for f in faithful_score]
avg_faithfulness = statistics.mean(score_faithful)

In [ ]:
zero_rate_all = [r["hit_graceful_degradation"] for r in results]

zero_rate = zero_rate_all.count(True)/len(zero_rate_all)

In [40]:
from datetime import datetime

metrics_baseline = {
    "run_timestamp": datetime.now().isoformat()  , 
    "avg_faithfulness": avg_faithfulness,
    "p50_latency_ms": round(p50),
    "p95_latency_ms": round(p95),
    "zero_result_rate": zero_rate
}
# metrics_baseline
with open("metrics_baseline.json", "w") as f:
    json.dump(metrics_baseline, f, indent=2)

print(json.dumps(metrics_baseline, indent=2))

{
  "run_timestamp": "2026-05-23T01:40:06.658212",
  "avg_faithfulness": 0.9183333333333333,
  "p50_latency_ms": 4261,
  "p95_latency_ms": 7842,
  "zero_result_rate": 0.0
}
